# VytalLink Health Chat Demo

End-to-end hackathon notebook: load real VytalLink data, compute readiness, generate an LLM narrative, and chat about the same health window.

## Observability First

Recommended demo order:

1. Start the local telemetry stack with `make obs-up`.
2. Open Grafana at `http://localhost:3000` and keep the `VytalLink App` dashboard visible.
3. Optionally open Jaeger at `http://localhost:16686` and LangSmith for traces.
4. Run this notebook from top to bottom.
5. Watch VytalLink request metrics during fetch, then LLM metrics during the narrative and chat sections.

If `OTEL_EXPORTER_OTLP_ENDPOINT` is configured but the collector is not running, the app will skip OTEL export and continue normally.

## 1. Setup

In [ ]:
from datetime import date
from dataclasses import dataclass

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import Markdown, display

from vytallink_health_kit.application.use_cases import (
    BuildReadinessReportInput,
    BuildReadinessReportUseCase,
    ChatWithHealthDataInput,
    ChatWithHealthDataUseCase,
)
from vytallink_health_kit.infrastructure.llm import LLMNarrativeGenerator
from vytallink_health_kit.infrastructure.observability import initialize_observability
from vytallink_health_kit.infrastructure.settings import (
    load_llm_settings,
    load_observability_settings,
    load_vytallink_settings,
)
from vytallink_health_kit.infrastructure.vytallink_client import VytalLinkRESTClient
from vytallink_health_kit.infrastructure.vytallink_client import VytalLinkClientError
from vytallink_health_kit.domain.entities import HealthData

load_dotenv()
obs_settings = initialize_observability(load_observability_settings())
vytallink_settings = load_vytallink_settings()
llm_settings = load_llm_settings()

provider = VytalLinkRESTClient(settings=vytallink_settings)
generator = LLMNarrativeGenerator(settings=llm_settings)
readiness_use_case = BuildReadinessReportUseCase(
    health_data_provider=provider,
    narrative_generator=generator,
)
chat_use_case = ChatWithHealthDataUseCase(
    health_data_provider=provider,
    narrative_generator=generator,
)

DEMO_END_DATE = date.today()
DEMO_DAYS = 7

@dataclass(slots=True)
class StaticHealthDataProvider:
    health_data: HealthData

    def fetch_window(self, *, end_date: date, days: int) -> HealthData:
        return self.health_data

print(
    {
        "base_url": vytallink_settings.base_url,
        "api_mode": vytallink_settings.api_mode,
        "llm_provider": llm_settings.llm_provider,
        "request_timeout_seconds": vytallink_settings.timeout_seconds,
        "metrics_timeout_seconds": vytallink_settings.metrics_timeout_seconds,
        "metrics_request_interval_seconds": vytallink_settings.metrics_request_interval_seconds,
        "otel_endpoint": obs_settings.otel_exporter_otlp_endpoint,
        "service_name": obs_settings.otel_service_name,
        "window": {"end_date": DEMO_END_DATE.isoformat(), "days": DEMO_DAYS},
    }
)

display(
    Markdown(
        """
### Live Telemetry Checklist

- Start the stack with `make obs-up`
- Grafana: `http://localhost:3000`
- Jaeger: `http://localhost:16686`
- Run the fetch cell and check VytalLink API traffic
- Run the narrative/chat cells and check LLM latency and traces
"""
    )
)


## 2. Fetch Data

In [ ]:
try:
    health_data = provider.fetch_window(end_date=DEMO_END_DATE, days=DEMO_DAYS)
except VytalLinkClientError as exc:
    raise RuntimeError(
        "VytalLink data fetch failed. The server may be saturated. "
        "Retry the cell in a moment or increase VYTALLINK_METRICS_TIMEOUT_SECONDS in .env. "
        f"Original error: {exc}"
    ) from exc

print(
    {
        "available_days": health_data.available_days,
        "missing_days": [day.isoformat() for day in health_data.missing_days],
    }
)

rows = []
for day in health_data.days:
    iso_day = day.isoformat()
    sleep = health_data.sleep.get(iso_day)
    hr = health_data.heart_rate.get(iso_day)
    activity = health_data.activity.get(iso_day)
    rows.append(
        {
            "date": day,
            "sleep_total_minutes": getattr(sleep, "total_minutes", None),
            "sleep_awake_minutes": getattr(sleep, "awake_minutes", None),
            "resting_hr_bpm": getattr(hr, "resting_bpm", None),
            "activity_steps": getattr(activity, "steps", None),
            "activity_calories": getattr(activity, "active_calories", None),
            "activity_exercise_minutes": getattr(activity, "exercise_minutes", None),
        }
    )

health_df = pd.DataFrame(rows)

static_provider = StaticHealthDataProvider(health_data=health_data)
readiness_use_case = BuildReadinessReportUseCase(
    health_data_provider=static_provider,
    narrative_generator=generator,
)
chat_use_case = ChatWithHealthDataUseCase(
    health_data_provider=static_provider,
    narrative_generator=generator,
)

print("Health data cached in-memory for the rest of the notebook.")
health_df


## 3. Visualize

In [ ]:
sns.set_theme(style="whitegrid")
plot_df = health_df.copy()
plot_df["date_label"] = pd.to_datetime(plot_df["date"]).dt.strftime("%Y-%m-%d")

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

sns.lineplot(
    data=plot_df,
    x="date_label",
    y="sleep_total_minutes",
    marker="o",
    sort=False,
    ax=axes[0],
    color="#235789",
)
axes[0].set_title("Sleep Trend")
axes[0].set_ylabel("Minutes")

sns.lineplot(
    data=plot_df,
    x="date_label",
    y="resting_hr_bpm",
    marker="o",
    sort=False,
    ax=axes[1],
    color="#c1292e",
)
axes[1].set_title("Resting Heart Rate Trend")
axes[1].set_ylabel("BPM")

sns.barplot(
    data=plot_df,
    x="date_label",
    y="activity_steps",
    ax=axes[2],
    color="#f1d302",
)
axes[2].set_title("Activity Steps")
axes[2].set_ylabel("Steps")
axes[2].set_xlabel("Date")

for axis in axes:
    axis.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 4. Readiness

In [ ]:
readiness_report = readiness_use_case.execute(
    BuildReadinessReportInput(
        end_date=DEMO_END_DATE,
        days=DEMO_DAYS,
        include_narrative=False,
    )
)

display(readiness_report.readiness)
display(Markdown(readiness_report.markdown))


## 5. Narrative LLM

In [ ]:
llm_report = readiness_use_case.execute(
    BuildReadinessReportInput(
        end_date=DEMO_END_DATE,
        days=DEMO_DAYS,
        include_narrative=True,
    )
)

display(Markdown(llm_report.narrative))


## 6. Interactive Chat

In [ ]:
print("Ask questions about your readiness, sleep, activity, or heart rate.")
print("Type 'exit' to stop the loop.")

while True:
    question = input("You: ").strip()
    if question.lower() in {"exit", "quit", "q"}:
        print("Chat ended.")
        break
    if not question:
        continue

    answer = chat_use_case.execute(
        ChatWithHealthDataInput(
            end_date=DEMO_END_DATE,
            days=DEMO_DAYS,
            question=question,
        )
    )
    print(f"Assistant: {answer}\n")
